In [45]:
import pandas as pd

In [46]:
pd.set_option("display.max_columns", 30)

In [47]:
df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/silver/flight_information/flights_info_silver.parquet")

## Feature Extraction 

### Departure Lat & Lon & Altitude

In [48]:
airport_df = pd.read_csv("/Users/YGT/ist-airport-decision-support-system/data/meta/airportdata.txt")

In [50]:
airport_df.shape

(84654, 19)

In [ ]:
airport_df = airport_df[["name", "latitude_deg","longitude_deg", "elevation_ft", "icao_code"]]

In [61]:
set_data = set(df["dep_code_icao"].unique())
set_airports = set(airport_df["icao_code"].unique())

# Kesişim (Eşleşenler)
matched = set_data.intersection(set_airports)

# Fark (Eşleşmeyenler - senin verinde olup kütüphanede olmayanlar)
missing = set_data.difference(set_airports)

print("="*40)
print(f"Flight İnfo Dataset Unique ICAO: {len(set_data):,}")
print(f"Airport Dataset Unique ICAO: {len(set_airports):,}")
print("-"*40)
print(f"Matched ICAO Count: {len(matched):,}")
print(f"Missing ICAO Count: {len(missing):,}")
print(f"Matched Ratio: %{(len(matched)/len(set_data))*100:.2f}")
print("="*40)

if len(missing) > 0:
    print("\n not matched icao code :")
    print(list(missing))

Flight İnfo Dataset Unique ICAO: 341
Airport Dataset Unique ICAO: 9,730
----------------------------------------
Matched ICAO Count: 328
Missing ICAO Count: 13
Matched Ratio: %96.19

 not matched icao code :
['UTFF', 'OKBK', 'UTSB', 'HEBA', 'UTSS', 'HSSJ', 'LTST', 'UTTT', 'UTNU', 'UTFA', 'UAFM', 'LERJ', 'OSNH']


In [63]:
airport_df.columns

Index(['name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'icao_code'], dtype='str')

In [65]:
airport_lookup = airport_df.copy()

df = pd.merge(
    df,
    airport_lookup,
    left_on='dep_code_icao',
    right_on='icao_code',
    how='left'
)

In [67]:
df.head(2)

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,airline_icao,dep_code_iata,dep_code_icao,dep_name_english,dest_code_iata,dest_code_icao,dest_name_english,dest_lat,dest_lon,dest_altitude,arr_sched_time_utc,arr_revised_time_utc,status,name,latitude_deg,longitude_deg,elevation_ft,icao_code
0,2025-07-31,Thursday,21,728678,Airbus A320,YI-ASX,I A W,IA 223,IAW223,IA,IAW,BGW,ORBI,Baghdad,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:00:00+00:00,2025-07-31 18:00:00+00:00,Arrived,Baghdad International Airport / New Al Muthana...,33.262501,44.234600,114.0,ORBI
1,2025-07-31,Thursday,21,7114EF,Airbus A321,UNKNOWN,Saudi Arabian,SV 261,SVA261,SV,SVA,JED,OEJN,Jeddah,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:10:00+00:00,2025-07-31 18:10:00+00:00,Arrived,King Abdulaziz International Airport,21.680241,39.157436,48.0,OEJN


In [70]:
df['dep_lat'] = df['latitude_deg'].astype(float)
df['dep_lon'] = df['longitude_deg'].astype(float)
df['dep_altitude'] = df['elevation_ft'].astype(float) #ft

In [74]:
cols_to_drop = ['name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'icao_code']

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [78]:
df.loc[
    (df["dep_lat"] == 0.0) &
    (df["dep_lon"] == 0.0)

]

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,airline_icao,dep_code_iata,dep_code_icao,dep_name_english,dest_code_iata,dest_code_icao,dest_name_english,dest_lat,dest_lon,dest_altitude,arr_sched_time_utc,arr_revised_time_utc,status,dep_lat,dep_lon,dep_altitude


In [ ]:
['UTFF', 'OKBK', 'UTSB', 'HEBA', 'UTSS', 'HSSJ', 'LTST', 'UTTT', 'UTNU', 'UTFA', 'UAFM', 'LERJ', 'OSNH']

In [90]:
AIRPORT_PATCH = [
    {"icao": "UTFF", "lat": 37.2867, "lon": 67.3100, "alt": 1017.0},
    {"icao": "UTNU", "lat": 42.4872, "lon": 59.6231, "alt": 250.0},
    {"icao": "OSNH", "lat": 37.0219, "lon": 41.1933, "alt": 1483.0},
    {"icao": "HSSJ", "lat": 4.8720,  "lon": 31.6011, "alt": 1457.0},
    {"icao": "LTST", "lat": 37.3647, "lon": 42.0583, "alt": 2038.0},
    {"icao": "UTTT", "lat": 41.2579, "lon": 69.2812, "alt": 1417.0},
    {"icao": "OKBK", "lat": 29.2266, "lon": 47.9689, "alt": 206.0},
    {"icao": "UTSB", "lat": 39.7750, "lon": 64.4833, "alt": 751.0},
    {"icao": "HEBA", "lat": 30.9177, "lon": 29.6964, "alt": 177.0},
    {"icao": "UTSS", "lat": 39.7005, "lon": 66.9838, "alt": 2224.0},
    {"icao": "UTFA", "lat": 40.7269, "lon": 72.2925, "alt": 1558.0},
    {"icao": "UAFM", "lat": 43.0613, "lon": 74.4776, "alt": 2058.0},
    {"icao": "LERJ", "lat": 42.4606, "lon": -2.3206, "alt": 1156.0},
]

In [93]:
patch_df = pd.DataFrame(AIRPORT_PATCH).copy()

patch_df = patch_df.rename(columns={
    "icao": "dep_code_icao",
    "lat": "dep_lat_patch",
    "lon": "dep_lon_patch",
    "alt": "dep_altitude_patch",
})


patch_df = patch_df[["dep_code_icao", "dep_lat_patch", "dep_lon_patch", "dep_altitude_patch"]]

df = df.merge(patch_df, on="dep_code_icao", how="left")

# fill only missing
df["dep_lat"] = df["dep_lat"].fillna(df["dep_lat_patch"])
df["dep_lon"] = df["dep_lon"].fillna(df["dep_lon_patch"])
df["dep_altitude"] = df["dep_altitude"].fillna(df["dep_altitude_patch"])

df = df.drop(columns=["dep_lat_patch", "dep_lon_patch", "dep_altitude_patch"])

In [94]:
df.isna().sum()

date                      0
day_of_week               0
hour_ist                  0
hex_icao                  0
aircraft_type             0
aircraft_registration     0
airline_name_english      0
callsign_code_iata        0
callsign_code_icao        0
airline_iata              0
airline_icao              0
dep_code_iata             0
dep_code_icao             0
dep_name_english          0
dest_code_iata            0
dest_code_icao            0
dest_name_english         0
dest_lat                  0
dest_lon                  0
dest_altitude             0
arr_sched_time_utc        0
arr_revised_time_utc     38
status                    0
dep_lat                   0
dep_lon                   0
dep_altitude              0
dtype: int64

In [95]:
df.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude'],
      dtype='str')